# Notebook 03: Heterogeneous Vehicle Fleet Demand Modeling & OD Matrix Generation

**Objective:** Project spatial EV charging demand ($D_i$) across Dhaka's traffic centroids for heterogeneous vehicle typologies (E2W, Easy-Bikes, Private 4W, Taxis, and E-Buses), and compute shortest path travel times with congestion factors.

In [ ]:
import os
import sys
import yaml
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.optimization.demand_model import DhakaEVDemandModel
from src.spatial.osm_network import compute_od_matrices

config_path = os.path.join(module_path, 'configs', 'default_config.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

demand_model = DhakaEVDemandModel(config)
city_totals = demand_model.calculate_total_city_demand()

print('City-Wide Daily EV Charging Demand Breakdown (Dhaka):')
for v_type, data in city_totals.items():
    if isinstance(data, dict):
        print(f"  - {v_type:15s}: {data['count']:,} vehicles | {data['total_daily_mwh']:.1f} MWh/day (Fast: {data['fast_charge_mwh']:.1f} MWh, Swap: {data['swap_mwh']:.1f} MWh, AC: {data['ac_mwh']:.1f} MWh)")
print(f"Total Metropolitan Daily Energy: {city_totals['total_city_daily_mwh']:.1f} MWh/day")

## 1. Inspecting Origin-Destination (OD) Travel Time & Distance Matrices

In [ ]:
od_path = os.path.join(module_path, 'data', 'processed', 'od_travel_time_matrix.npz')
data = np.load(od_path)
dist_mat = data['distances']
time_mat = data['travel_times']

print(f'OD Matrix Shape: {dist_mat.shape[0]} demand zones x {dist_mat.shape[1]} candidate sites')
print(f'Mean Distance: {np.mean(dist_mat)/1000.0:.2f} km | Max Distance: {np.max(dist_mat)/1000.0:.2f} km')
print(f'Mean Congestion Travel Time: {np.mean(time_mat):.1f} min | Max Travel Time: {np.max(time_mat):.1f} min')